# SME Voice Assistant - Evaluation
**Karan Homayounfar - UWE Bristol MSc Data Science IGP**

Compares fine-tuned vs vanilla for Phi-3 mini and Llama 3.2 3B.

Before running:
1. GPU T4 x2 enabled
2. Add dataset: `sme-voice-data1` (has sme_test.jsonl)
3. Add dataset: `sme-phi3-adapter` (adapter files from checkpoints/sme-phi3-qlora)
4. Add dataset: `sme-llama3-adapter` (adapter files from checkpoints/sme-llama3-qlora/checkpoints/sme-llama3-qlora)
5. Add HF_TOKEN secret for Llama 3

In [ ]:
import os, torch

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
!pip install -q "transformers==4.45.2" "trl==0.11.4" "peft>=0.11.0" "bitsandbytes==0.45.5" accelerate datasets numpy

In [ ]:
import json, time
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

TEST_FILE      = '/kaggle/input/datasets/karanhomayounfar1/sme-voice-data/sme_train.jsonl'
PHI3_ADAPTER   = '/kaggle/input/datasets/karanhomayounfar1/sme-phi3-adapter'
LLAMA3_ADAPTER = '/kaggle/input/datasets/karanhomayounfar1/sme-llama3-adapter'
OUTPUT_DIR     = '/kaggle/working/eval_results'


os.makedirs(OUTPUT_DIR, exist_ok=True)

ds = load_dataset('json', data_files=TEST_FILE, split='train')
records = [dict(r) for r in ds]
print(f'Test samples: {len(records)}')

In [ ]:
def get_bnb():
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )

def load_base(model_id, hf_token=None):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token=hf_token)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=hf_token, quantization_config=get_bnb(),
        device_map='auto', trust_remote_code=True,
        torch_dtype=torch.bfloat16, attn_implementation='eager',
    )
    model.eval()
    return tok, model

def load_adapter(base_model, adapter_path):
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model

def fmt_phi3(rec):
    return f"<|system|>\n{rec['instruction']}<|end|>\n<|user|>\n{rec['input']}<|end|>\n<|assistant|>\n"

def fmt_llama3(rec):
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f"{rec['instruction']}<|eot_id|>"
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f"{rec['input']}<|eot_id|>"
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )

def infer(model, tok, prompt, eos_token, max_new=60):
    inputs = tok(prompt, return_tensors='pt').to('cuda')
    input_len = inputs['input_ids'].shape[1]
    eos_id = tok.convert_tokens_to_ids(eos_token)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=eos_id)
    t1 = time.perf_counter()
    text = tok.decode(out[0][input_len:], skip_special_tokens=True).strip()
    return text, (t1 - t0) * 1000

def parse_json(text):
    s, e = text.find('{'), text.rfind('}')
    if s == -1 or e == -1:
        return None
    try:
        return json.loads(text[s:e+1])
    except:
        return None

def run_eval(label, records, fmt_fn, infer_fn):
    results = []
    for i, rec in enumerate(records):
        prompt = fmt_fn(rec)
        expected = json.loads(rec['output'])
        text, lat = infer_fn(prompt)
        predicted = parse_json(text)
        json_valid = predicted is not None
        action_ok = json_valid and predicted.get('action') == expected.get('action')
        fields_ok = action_ok and all(
            predicted.get(k) == v for k, v in expected.items() if k != 'action'
        )
        exact = fields_ok and set(predicted.keys()) == set(expected.keys())
        results.append({'json_valid': json_valid, 'action_correct': action_ok,
                        'fields_correct': fields_ok, 'exact_match': exact, 'latency_ms': lat,
                        'expected': expected, 'predicted': predicted, 'raw': text})
        if (i + 1) % 10 == 0:
            print(f'  {label}: {i+1}/{len(records)}')

    n = len(results)
    lats = np.array([r['latency_ms'] for r in results])
    summary = {
        'label': label, 'n': n,
        'json_valid_rate':  round(sum(r['json_valid']      for r in results) / n, 4),
        'action_accuracy':  round(sum(r['action_correct']  for r in results) / n, 4),
        'field_accuracy':   round(sum(r['fields_correct']  for r in results) / n, 4),
        'exact_match_rate': round(sum(r['exact_match']     for r in results) / n, 4),
        'latency_ms': {'mean': round(float(lats.mean()), 1),
                       'p50':  round(float(np.percentile(lats, 50)), 1),
                       'p95':  round(float(np.percentile(lats, 95)), 1)},
    }
    return summary, results

print('Helpers ready.')

In [ ]:
PHI3_ID = 'microsoft/Phi-3-mini-4k-instruct'

print('Loading Phi-3 vanilla...')
phi3_tok, phi3_vanilla = load_base(PHI3_ID)

print('Evaluating vanilla Phi-3...')
vanilla_summary, vanilla_results = run_eval(
    'vanilla_phi3', records,
    fmt_phi3,
    lambda p: infer(phi3_vanilla, phi3_tok, p, '<|end|>')
)
print(vanilla_summary)

In [ ]:
print('Loading Phi-3 fine-tuned adapter...')
phi3_finetuned = load_adapter(phi3_vanilla, PHI3_ADAPTER)

print('Evaluating fine-tuned Phi-3...')
finetuned_summary, finetuned_results = run_eval(
    'finetuned_phi3_qlora', records,
    fmt_phi3,
    lambda p: infer(phi3_finetuned, phi3_tok, p, '<|end|>')
)
print(finetuned_summary)

del phi3_vanilla, phi3_finetuned, phi3_tok
torch.cuda.empty_cache()

In [ ]:
LLAMA3_ID = 'meta-llama/Llama-3.2-3B-Instruct'

print('Loading Llama 3 vanilla...')
llama_tok, llama_vanilla = load_base(LLAMA3_ID, HF_TOKEN)

print('Evaluating vanilla Llama 3...')
llama_vanilla_summary, llama_vanilla_results = run_eval(
    'vanilla_llama3', records,
    fmt_llama3,
    lambda p: infer(llama_vanilla, llama_tok, p, '<|eot_id|>')
)
print(llama_vanilla_summary)

In [ ]:
print('Loading Llama 3 fine-tuned adapter...')
llama_finetuned = load_adapter(llama_vanilla, LLAMA3_ADAPTER)

print('Evaluating fine-tuned Llama 3...')
llama_ft_summary, llama_ft_results = run_eval(
    'finetuned_llama3_qlora', records,
    fmt_llama3,
    lambda p: infer(llama_finetuned, llama_tok, p, '<|eot_id|>')
)
print(llama_ft_summary)

del llama_vanilla, llama_finetuned, llama_tok
torch.cuda.empty_cache()

In [ ]:
all_summaries = [vanilla_summary, finetuned_summary, llama_vanilla_summary, llama_ft_summary]

print(f"\n{'Metric':<25} {'vanilla_phi3':<20} {'finetuned_phi3':<20} {'vanilla_llama3':<20} {'finetuned_llama3':<20}")
print('-' * 85)
for metric in ['json_valid_rate', 'action_accuracy', 'field_accuracy', 'exact_match_rate']:
    row = f"{metric:<25}"
    for s in all_summaries:
        row += f"{s[metric]:.1%}{'':>13}"
    print(row)
for pct in ['mean', 'p50', 'p95']:
    row = f"{'latency_' + pct + '_ms':<25}"
    for s in all_summaries:
        row += f"{s['latency_ms'][pct]:<20}"
    print(row)

with open(f'{OUTPUT_DIR}/eval_summary.json', 'w') as f:
    json.dump(all_summaries, f, indent=2)

for s, r in [('vanilla_phi3', vanilla_results), ('finetuned_phi3', finetuned_results),
             ('vanilla_llama3', llama_vanilla_results), ('finetuned_llama3', llama_ft_results)]:
    with open(f'{OUTPUT_DIR}/eval_{s}.json', 'w') as f:
        json.dump(r, f, indent=2)

print(f'\nResults saved to {OUTPUT_DIR}')